In [ ]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from IPython.display import clear_output
%matplotlib inline

np.random.seed(42)

# PyTorch 简介

PyTorch 是目前最著名的神经网络搭建框架之一（本课程要做的就是搭建神经网络）。

它最显而易见的替代品是 TensorFlow，不过就当前（2018 年秋）而言，TensorFlow 对用户远没有那么友好，所以我们还是用 PyTorch。

而且说真的，只要你学会了其中一个，另一个自然也就学得会。

## 自动微分（Autograd）

我们先从 PyTorch 最核心的概念之一讲起——自动微分（automatic differentiation，即 autograd）。

### 计算图

计算图（computational graph）提供了一种非常方便的方式来表示函数并计算它们的梯度。

例如，
$$f = (x + y) \cdot z$$

可以用下面这张图来表示：  
![graph](https://github.com/DanAnastasyev/DeepNLP-Course/raw/master/Week%2003/Images/Circuit.png)  
*来自 [Backpropagation, Intuitions - CS231n](http://cs231n.github.io/optimization-2/)*

**前向传播（forward pass）**计算函数的值（图中的绿色数字）。它从输入（左侧）出发，依次施加一连串函数。

**反向传播（backward pass / back propagation）**则用来计算函数的梯度，在我们的例子里就是 $\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}, \frac{\partial f}{\partial z}$。它从输出出发，运用**链式法则（chain rule）**把梯度算出来。

例如对于 $f = q \cdot z$，有 $\frac{\partial f}{\partial q} = z$，$\frac{\partial f}{\partial z} = q$。  
对于 $q = x + y$，有 $\frac{\partial q}{\partial x} = \frac{\partial q}{\partial y} = 1$。  
最后套用链式法则：$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial q} \frac{\partial q}{\partial x} = z$。

*如果上面这些内容你理解起来有困难（就算没有困难也一样），推荐看这篇很棒的教程：[Backpropagation, Intuitions - CS231n](http://cs231n.github.io/optimization-2/)。*

而在 PyTorch 里做这类计算相当简单。你只需要把函数描述成一连串的运算，比如：

In [ ]:
x = torch.tensor(-2., requires_grad=True)
y = torch.tensor(5., requires_grad=True)
z = torch.tensor(-4., requires_grad=True)

q = x + y
f = q * z

传入一些参数，然后对它说一句：“嘿，把你的梯度算出来吧。”于是魔法就发生了：

![graph](https://raw.githubusercontent.com/pytorch/pytorch/master/docs/source/_static/img/dynamic_graph.gif)  
*来自 [github.com/pytorch/pytorch](https://github.com/pytorch/pytorch)*

PyTorch 会自行构建计算图并完成反向传播。

如果你已经熟悉 TensorFlow，就会看到最主要的区别：这里的图是**动态**构建的，不需要先编译之类的东西。

基本上，这意味着你调试代码会容易得多。

In [ ]:
f.backward()

print('df/dz =', z.grad)
print('df/dx =', x.grad)
print('df/dy =', y.grad)

调用 `backward()` 方法会为计算图中的所有张量计算梯度（`requires_grad == False` 的子图除外）。

*想深入了解 PyTorch 的 autograd，可以读这里：[Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html)。*

PyTorch 最讨人喜欢的一点是：你完全可以像用 numpy 那样用它。你一直都在用 `ndarray`，对吧？

它有一个对应的东西——`tensor`（张量）。我们其实刚刚已经创建了好几个。

它像 `ndarray` 一样保存数据：

In [ ]:
x.data

还能保存梯度（这一点与 `ndarray` 不同）：

In [ ]:
x.grad

以及用于计算该梯度所调用的函数：

In [ ]:
q.grad_fn

还有一大堆元信息：

In [ ]:
x.type(), x.shape, x.device, x.layout

想更多了解张量，请看这个教程：[Deep Learning with PyTorch: A 60 Minute Blitz > What is PyTorch?](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html#sphx-glr-beginner-blitz-tensor-tutorial-py)

有时候我们并不想计算梯度。为了处理这类情况（具体例子我们很快就会讨论），可以使用上下文管理器（[Locally disabling gradient computation](https://pytorch.org/docs/stable/autograd.html#locally-disabling-gradient-computation)）：
```python
torch.autograd.no_grad()
torch.autograd.enable_grad()
torch.autograd.set_grad_enabled(mode)
```

In [ ]:
with torch.autograd.no_grad():
    x = torch.tensor(-2., requires_grad=True)
    y = torch.tensor(5., requires_grad=True)
    q = x + y

z = torch.tensor(-4., requires_grad=True)
f = q * z

f.backward()

print('df/dz =', z.grad)
print('df/dx =', x.grad)
print('df/dy =', y.grad)

那么问题来了：你到底为什么会想用它呢 :)

### 热身任务

为了理解它，我们先写一个简单的线性回归实现：

In [ ]:
w_orig, b_orig = 2.6, -0.4

X = np.random.rand(100) * 10. - 5.
y_orig = w_orig * X + b_orig

y = y_orig + np.random.randn(100)

plt.plot(X, y, '.')
plt.plot(X, y_orig)
plt.show()

这里有两个参数 $w$ 和 $b$。我们希望它们尽可能接近 $w_{orig}, b_{orig}$。

那我们要优化什么呢？就来优化 MSE 损失吧：
$$J(w, b) = \frac{1}{N} \sum_{i=1}^N || \hat y_i - y_i(w, b)||^2 =\frac{1}{N} \sum_{i=1}^N || \hat y_i - (w \cdot x_i + b)||^2. $$

我们可以用*梯度下降*算法来优化它（现在甚至还不是随机梯度下降）：
$$w_{t+1} := w_t - \alpha \cdot \frac{\partial J}{\partial w}(w_t, b_t)$$
$$b_{t+1} := w_t - \alpha \cdot \frac{\partial J}{\partial b}(w_t, b_t)$$

*你看，这里就很适合用反向传播。*

**任务** 用纯 numpy 实现这个优化过程。

你需要：
1. 完成前向传播：$y(w, b) = w \cdot x + b$；
2. 用反向传播求出梯度 $\frac{\partial J}{\partial w}, \frac{\partial J}{\partial b}$；
3. 让 $w, b$ 沿负梯度方向移动。

> 译注：原文 $b$ 的更新式写成了 $w_t$，按梯度下降应为 $b_{t+1} := b_t - \alpha \cdot \frac{\partial J}{\partial b}(w_t, b_t)$，此处保留原文写法以便对照。

In [ ]:
def display_progress(epoch, loss, w, b, X, y, y_pred):
    clear_output(True)
    print('Epoch = {}, Loss = {}, w = {}, b = {}'.format(epoch, loss, w, b))
    plt.plot(X, y, '.')
    plt.plot(X, y_pred)
    plt.show()
    time.sleep(1)


w = np.random.randn()
b = np.random.randn()

alpha = 0.01

for i in range(100):
    <计算模型的输出>

    <计算损失>

    <计算梯度>

    <更新 w 和 b>

    if (i + 1) % 5 == 0:
        display_progress(i + 1, loss, w, b, X, y, y_pred)

assert np.abs(w - w_orig) < 0.1, 'Something went wrong :('

用 PyTorch 实现同样的事情要简单得多。

前向传播看起来几乎一模一样。而反向传播我们已经学会怎么做了！只要调用 `loss.backward()`。

不过有几个坑你必须知道。

首先，`w` 和 `b` 可不是随随便便就能更新的。你可以试试：

In [ ]:
w = torch.randn(1, requires_grad=True)

w -= 1.

它应该会报一个类似这样的错：  
`RuntimeError: a leaf Variable that requires grad has been used in an in-place operation.`

问题出在 autograd 对原地操作（in-place operation）的支持上：[In place operations with autograd](https://pytorch.org/docs/stable/notes/autograd.html#in-place-operations-with-autograd)。

但实际上，我们要做的这个操作并不需要梯度，我们只是想更新这个节点的取值而已。

要绕开这个问题，我们可以用 `no_grad` 上下文，或者直接更新张量底层的数据：

In [ ]:
w.data -= 1.

另一件需要留意的事是：梯度默认会**累加**。所以每次调用 `loss.backward()` 之间，你得自己把它们清零：
```python
w.grad.zero_()
b.grad.zero_()
```

**任务** 用 PyTorch 实现线性回归。

In [ ]:
X = torch.as_tensor(X).float()
y = torch.as_tensor(y).float()

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

for i in range(100):
    <复制前向传播，并加上反向传播与参数更新>

    if (i + 1) % 5 == 0:
        display_progress(i + 1, loss, w.item(), b.item(),
                         X.data.numpy(), y.data.numpy(), y_pred.data.numpy())

是不是简单多了？:)